<a href="https://colab.research.google.com/github/tobiasllop/Tesis/blob/main/Parte_B_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Celda 1: Importación de librerías y Chunking (Parte B.1)

---



In [2]:
!pip install prince
!pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 73.3 MB/s eta 0:00:00


In [ ]:
# !pip install fastparquet # <-- Corré esto en una celda previa si no lo tenés

import pandas as pd
import gc
import os
import shutil
import fastparquet # Import fastparquet for efficient Parquet file handling

# 1. Definir las rutas
# NOTA: Si deudores_final_enero es realmente un .txt o .csv, poné bien la extensión.
# Si es un .parquet original no podés usar read_csv. Asumo que es un CSV separado por ';' como pusiste antes.
ruta_deudores_26 = '/content/drive/MyDrive/Tesis2026/deudores_final_enero.parquet'
ruta_padron = '/content/drive/MyDrive/Tesis2026/padron_fisicas_enero.parquet'

# Rutas de salida (Una local ultra rápida y la definitiva en tu Drive)
ruta_salida_local = '/content/cruce_final_enero.parquet'
ruta_salida_drive = '/content/drive/MyDrive/Tesis2026/cruce_final_enero.parquet'

# 2. Cargar el Padrón (Optimizando columnas)
print("Cargando padrón...")
# TRUCO: Definí solo las columnas que vas a cruzar y usar. Si necesitás más, agregalas a esta lista.
# Las columnas 'genero_deudor', 'edad_deudor', 'provincia_deudor' no existen en el archivo.
# Se han reemplazado por 'sexo' y 'provincia' que sí existen, y se ha eliminado 'edad_deudor'.
columnas_padron = ['cuit', 'sexo', 'provincia', 'fecha_nacimiento']
padron = pd.read_parquet(ruta_padron, columns=columnas_padron)

# Optimize 'cuit' in padron: try converting to numeric first for memory efficiency, then to string for merge
padron['cuit'] = pd.to_numeric(padron['cuit'], errors='coerce')
# Fill NaNs with a placeholder and convert to Int64 (if possible), then to string
padron['cuit'] = padron['cuit'].fillna(-1).astype('Int64').astype(str)

# Optimize 'sexo' and 'provincia' columns in padron to category dtype for memory efficiency
padron['sexo'] = padron['sexo'].astype('category')
padron['provincia'] = padron['provincia'].astype('category')
# Convert 'fecha_nacimiento' to datetime
padron['fecha_nacimiento'] = pd.to_datetime(padron['fecha_nacimiento'], errors='coerce')

# 3. Procesamiento por chunks
chunksize = 1000000
df_merged_list = [] # Ya no la usamos, pero la saco para no confundir

print("Iniciando procesamiento por chunks y guardando en disco local...")

try:
    # Initialize 'i' before the loop to ensure it's always defined for error messages
    i = -1

    # Use fastparquet to read the parquet file in chunks (row groups)
    parquet_file = fastparquet.ParquetFile(ruta_deudores_26)

    # Iterate through row groups directly using iter_row_groups()
    for i, chunk in enumerate(parquet_file.iter_row_groups()):

        # Rename 'nro_id' to 'cuit' in the chunk to match the padron DataFrame for merging
        if 'nro_id' in chunk.columns:
            chunk = chunk.rename(columns={'nro_id': 'cuit'})
        else:
            raise ValueError("Column 'nro_id' (expected CUIT equivalent) not found in deudores_final_enero.parquet chunk.")

        # Optimize 'cuit' in chunk: try converting to numeric first for memory efficiency, then to string for merge
        chunk['cuit'] = pd.to_numeric(chunk['cuit'], errors='coerce')
        # Fill NaNs with a placeholder and convert to Int64 (if possible), then to string
        chunk['cuit'] = chunk['cuit'].fillna(-1).astype('Int64').astype(str)

        # Inner join con el padrón usando el CUIT
        merged_chunk = chunk.merge(padron, on='cuit', how='inner')

        # Guardar el pedazo cruzado directo a disco local
        if i == 0:
            # El primer chunk crea el archivo
            merged_chunk.to_parquet(ruta_salida_local, engine='fastparquet', append=False)
        else:
            # Los siguientes se "pegan" al final del archivo
            merged_chunk.to_parquet(ruta_salida_local, engine='fastparquet', append=True)

        print(f"Chunk {i+1} cruzado y guardado localmente.")

        # LIMPIEZA DE RAM EXTREMA (Obligatorio)
        del merged_chunk
        del chunk
        gc.collect()

    print("Proceso de cruce finalizado. Copiando el archivo consolidado a Google Drive...")

    # 4. Mover el archivo final seguro a tu Drive
    shutil.copy2(ruta_salida_local, ruta_salida_drive)
    print(f"¡Éxito total! Archivo gigante guardado de forma segura en: {ruta_salida_drive}")

except Exception as e:
    # Check if 'i' was defined to give more specific error message
    chunk_info = f"en el chunk {i+1}" if i >= 0 else "al inicio del proceso de chunking"
    print(f"Se produjo un error {chunk_info}: {e}")

finally:
    # Opcional: Limpiar el disco de Colab para no ocupar espacio de sobra
    if os.path.exists(ruta_salida_local):
        os.remove(ruta_salida_local)
        print("Archivo temporal local eliminado para liberar espacio.")

Cargando padrón...


# Celda 2: Armado de Categorías

In [ ]:
# A. Tamaño de deuda (Usando Cuantiles - Tertiles)
# q=3 divide en Pequeña, Mediana, Grande asegurando igual cantidad de casos en cada balde
df_2026['Tamanio_Deuda'] = pd.qcut(df_2026['deuda_total'], q=3, labels=['Pequeña', 'Mediana', 'Grande'])

# B. Grupo Entidad
df_2026 = df_2026.merge(entidades[['cod_entidad', 'grupo_entidad']], on='cod_entidad', how='left')

# C. Género (Limpieza básica)
df_2026['Genero'] = df_2026['genero'].fillna('Desc')

# D. Rango Etario
bins_edad = [17, 25, 40, 65, 120]
labels_edad = ['Jóvenes (18-25)', 'Adultos en Inserción (26-40)', 'Adultos Consolidados (41-65)', 'Adultos Mayores (>65)']
df_2026['Rango_Etario'] = pd.cut(df_2026['edad'], bins=bins_edad, labels=labels_edad)

# E. Región (CABA, AMBA, Interior)
# Ajustá la lógica dependiendo de cómo se llame tu columna de provincia o código postal
def clasificar_region(prov):
    prov = str(prov).upper()
    if 'CABA' in prov or 'CAPITAL' in prov:
        return 'CABA'
    elif 'BUENOS AIRES' in prov:
        return 'AMBA/PBA' # Si tenés CP podrías separar AMBA estricto del interior de PBA
    else:
        return 'Interior'

df_2026['Region'] = df_2026['provincia'].apply(clasificar_region)

# Nos quedamos con las columnas categóricas limpias para el modelo
cols_categoricas = ['Tamanio_Deuda', 'grupo_entidad', 'Genero', 'Rango_Etario', 'Region']
df_mca_2026 = df_2026[cols_categoricas].dropna()

# Celda 3: Análisis Estadístico y Gráficos

In [ ]:
# Gráficos de barras para ver las proporciones
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(3, 2, figsize=(15, 15))
axes = axes.flatten()

for i, col in enumerate(cols_categoricas):
    sns.countplot(y=col, data=df_mca_2026, order=df_mca_2026[col].value_counts().index,
                  ax=axes[i], palette='magma')
    axes[i].set_title(f'Distribución de {col} - Enero 2026', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Cantidad de Deudas')
    axes[i].set_ylabel('')

# Ocultar el último gráfico vacío si hay impares
fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

# Celda 4: Múltiple Correspondencias (MCA)

In [ ]:
print("Entrenando el modelo MCA... (esto puede tardar unos minutos)")
mca_26 = prince.MCA(n_components=2, n_iter=3, random_state=42)
mca_26 = mca_26.fit(df_mca_2026)

# Graficar el mapa perceptual
fig, ax = plt.subplots(figsize=(12, 8))
mca_26.plot(
    df_mca_2026,
    x_component=0,
    y_component=1,
    show_row_markers=False,     # Apagamos los puntitos de los deudores individuales
    show_column_markers=True,   # Mostramos las categorías
    show_row_labels=False,
    show_column_labels=True,
    ax=ax
)
plt.title('Mapa Perceptual MCA - Sistema Financiero (Enero 2026)', fontsize=16)
plt.show()